<a href="https://colab.research.google.com/github/ocayaro/Ocaya-Yakuphanoglu-method-program/blob/main/varshni.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# %% [markdown]
# # Single-mode AHC bandgap: Lie-symmetry simulator
#
# Companion notebook to *"Lie-Symmetry Structure of the Single-Mode Electron–Phonon Bandgap Shift and the Emergence of Varshni's Relation as a Regularized High-Temperature Limit"*.
#
# Everything is built from the exact single-mode Bose–Einstein gap
# $E(T)=E_\infty-a\coth(\theta/2T)$, with $a=\alpha\theta/2$, and is organised in the article's order:
# Varshni's form appears only as an *approximation* of the exact curve.
#
# | View | What it shows | Article |
# |---|---|---|
# | 1 Overview | exact curve vs. the Varshni forms | Secs. 2, 7 |
# | 2 Parity | $P:(T,G)\mapsto(-T,-G)$ and the parity defect | Prop. 1, Prop. 5(iv) |
# | 3 Tail | deviation $f=E-E_\infty+\alpha T$ (Langevin), Euler part, $\varepsilon(T)$ | Props. 3–4 |
# | 4 Errors | $E_V-E$ for level-, tail-matched and fitted $c$ | Sec. 8 |
# | 5 Orbits | finite flows of $X_D=T\partial_T+G\partial_E$ and $X_P=T^2\partial_T$ | Thm. 1–2 |
# | 6 ODE residual | numerical test that $X_D$, $X_P$ are symmetries and $\partial_T$ is not | Thm. 1 |
# | 7 Series | Bernoulli (graded) tail series and its radius $T>\theta/2\pi$ | Prop. 3 |
# | 8 Continuation | integrate the ODE downward from truncated tail data | Secs. 6–7 |
#
# Run all cells, then use the controls under the last cell. Units: $T$ in K, $E$ in eV, $\alpha$ in eV/K.

# %%
# --- Setup ---------------------------------------------------------------
import io, os, sys, zipfile, warnings
from dataclasses import dataclass
from math import factorial

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar
from scipy.special import bernoulli
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import files as colab_files

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 9,
    "axes.grid": True,
    "grid.linestyle": ":",
    "grid.alpha": 0.6,
})

COL = {"exact": "tab:blue", "level": "tab:red", "tail": "tab:green",
       "user": "tab:purple", "aux": "tab:orange", "ref": "0.35"}

# %%
# --- Physics core (dimensionless: t = T/theta, g = (E - E_inf)/(alpha*theta)) -----
C_LEVEL = 0.5                      # level matching:  c = theta/2
C_TAIL = 1.0 / np.sqrt(12.0)       # tail matching:   c = theta/sqrt(12)
T_SERIES_RADIUS = 1.0 / (2.0 * np.pi)


@dataclass(frozen=True)
class Params:
    E_inf: float = 3.0       # eV, intercept of the classical asymptote
    alpha: float = 4.0e-4    # eV/K, classical (Dulong-Petit) slope
    theta: float = 300.0     # K, phonon temperature hbar*omega/k_B

    @property
    def A(self):    return self.alpha * self.theta     # eV, electron-phonon amplitude
    @property
    def a(self):    return 0.5 * self.A                # eV, E_inf - E(0)
    @property
    def lam(self):  return 2.0 / self.alpha            # K/eV, ODE parameter lambda
    @property
    def E0(self):   return self.E_inf - self.a         # eV, E(T=0)
    def E_of_g(self, g): return self.E_inf + self.A * np.asarray(g)
    def E_of_D(self, D): return self.E_inf - self.A * np.asarray(D)


def _arr(t):
    return np.asarray(t, dtype=float)


def D_exact(t):
    """(E_inf - E)/(alpha*theta) = (1/2) coth(1/2t); odd in t."""
    t = _arr(t)
    with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
        return 0.5 / np.tanh(0.5 / t)

def g_exact(t):          return -D_exact(t)
def f_exact(t):          t = _arr(t); return -(D_exact(t) - t)           # Langevin deviation
def epsilon(t):          t = _arr(t); return 1.0 / (12.0 * t**2)          # nonlinearity / linear term

def D_varshni(t, c):     t = _arr(t); return t + c**2 / (t + c)           # asymptote-matched Varshni
def g_varshni(t, c):     return -D_varshni(t, c)
def f_varshni(t, c):     t = _arr(t); return -c**2 / (t + c)

def _series_coeffs(K):
    B = bernoulli(2 * K)                                                  # B_0 .. B_2K
    return [B[2 * k] / factorial(2 * k) for k in range(K + 1)]

def D_series(t, K):
    """Partial sum  sum_{k<=K} B_2k t^(1-2k)/(2k)!  (converges for t > 1/2pi)."""
    t = _arr(t); co = _series_coeffs(K)
    return sum(co[k] * t ** (1 - 2 * k) for k in range(K + 1))

def dDdt_series(t, K):
    t = _arr(t); co = _series_coeffs(K)
    return sum(co[k] * (1 - 2 * k) * t ** (-2 * k) for k in range(K + 1))


# --- Finite group actions on a solution g(t) (Theorems 1-2) ---------------
def act_dilation(g, tau):
    """Flow of X_D = T d_T + G d_E:  g -> e^tau g(t e^-tau)  (theta -> theta e^tau at fixed alpha)."""
    s = np.exp(tau)
    return lambda t: s * g(_arr(t) / s)

def act_projective(g, eps):
    """Flow of X_P = T^2 d_T:  1/T -> 1/T - eps, i.e. a shift of nu_0 (fugacity)."""
    def h(t):
        t = _arr(t)
        with np.errstate(divide="ignore", invalid="ignore"):
            return g(1.0 / (1.0 / t + eps))
    return h

def act_translation(g, c):
    """T -> T + c  (NOT a symmetry of the exact equation)."""
    return lambda t: g(_arr(t) + c)

def act_translation_shift(g, c):
    """T -> T + c combined with an E-shift (approximate symmetry at large T only)."""
    return lambda t: g(_arr(t) + c) + c


def ode_residual(g, t, h_rel=1e-3):
    """Relative residual of  t^2 g'' + 2 t g' + 2 g g' = 0  (dimensionless form of E'' = Phi E').
    Five-point finite differences; accurate to ~1e-7 for t >~ 0.15."""
    t = _arr(t); h = h_rel * t
    g0, gp1, gm1, gp2, gm2 = g(t), g(t + h), g(t - h), g(t + 2 * h), g(t - 2 * h)
    g1 = (-gp2 + 8 * gp1 - 8 * gm1 + gm2) / (12 * h)
    g2 = (-gp2 + 16 * gp1 - 30 * g0 + 16 * gm1 - gm2) / (12 * h**2)
    R = t**2 * g2 + 2 * t * g1 + 2 * g0 * g1
    scale = np.abs(t**2 * g2) + np.abs(2 * t * g1) + np.abs(2 * g0 * g1)
    return R / np.maximum(scale, 1e-300)


# --- Fits and statistics --------------------------------------------------
def fit_c(t_lo, t_hi, n=4000):
    """Least-squares c (units of theta) for the asymptote-matched Varshni form on [t_lo, t_hi]."""
    ts = np.linspace(t_lo, t_hi, n)
    obj = lambda c: np.sqrt(np.mean((D_varshni(ts, c) - D_exact(ts)) ** 2))
    r = minimize_scalar(obj, bounds=(0.02, 2.0), method="bounded", options={"xatol": 1e-10})
    return float(r.x), float(r.fun)

def error_stats(c, t_lo, t_hi, n=4000):
    ts = np.linspace(t_lo, t_hi, n)
    d = D_varshni(ts, c) - D_exact(ts)
    i = np.argmax(np.abs(d))
    return float(np.sqrt(np.mean(d**2))), float(np.abs(d[i])), float(ts[i])


# --- Downward continuation of the ODE from truncated tail data ------------
def continue_from_tail(t0, K, t_end, n=500):
    """Integrate g_nunu = 2 g g_nu (nu = theta/T) from tail data of order K at t0 down to t_end."""
    nu0, nu1 = 1.0 / t0, 1.0 / t_end
    y0 = [-float(D_series(t0, K)), float(t0**2 * dDdt_series(t0, K))]
    blow = lambda nu, y: abs(y[0]) - 1e3
    blow.terminal = True
    sol = solve_ivp(lambda nu, y: [y[1], 2 * y[0] * y[1]], (nu0, nu1), y0,
                    rtol=1e-11, atol=1e-13, dense_output=True, events=blow)
    nus = np.linspace(nu0, sol.t[-1], n)
    return 1.0 / nus, sol.sol(nus)[0]

# %%
# --- Self-tests: the numbers quoted in the article -----------------------------
def run_self_tests(verbose=True):
    t = np.linspace(0.15, 4.0, 400)
    r_exact = np.max(np.abs(ode_residual(g_exact, t)))
    r_D = np.max(np.abs(ode_residual(act_dilation(g_exact, 0.4), t)))
    r_P = np.max(np.abs(ode_residual(act_projective(g_exact, 0.7), t)))
    r_T = np.max(np.abs(ode_residual(act_translation(g_exact, 0.3), t)))
    assert r_exact < 1e-4 and r_D < 1e-4 and r_P < 1e-4, (r_exact, r_D, r_P)      # symmetries
    assert r_T > 1e-2, r_T                                                         # d_T is not one
    assert np.max(np.abs(D_exact(t) + D_exact(-t))) < 1e-15                        # parity
    for c in (C_LEVEL, C_TAIL):                                                    # Thm. 3 / Nernst
        h = 1e-6
        assert abs((D_varshni(h, c) - D_varshni(0.0, c)) / h) < 5e-6 * 10          # E'(0) = 0
        assert abs(D_varshni(0.0, c) - c) < 1e-14                                  # E_V(0) = E_inf - alpha*c
    assert abs(D_exact(0.0 + 1e-3) - 0.5) < 1e-12                                  # E(0) = E_inf - a
    assert abs(D_series(1.0, 6) - D_exact(1.0)) < 1e-6                             # Bernoulli series
    rho_exact = 0.5**2 / (1.0 * (1.0 / 12.0)); rho_V = (C_LEVEL**2) / (C_LEVEL**2)
    assert abs(rho_exact - 3.0) < 1e-12 and abs(rho_V - 1.0) < 1e-12               # rho = 3 vs 1
    tt, gg = continue_from_tail(6.0, 3, 0.4)                                       # ODE integration
    assert np.max(np.abs(gg - g_exact(tt))) < 1e-4
    if verbose:
        print("All self-tests passed.")
        print("  max ODE residual  exact: %.1e   X_D image: %.1e   X_P image: %.1e" % (r_exact, r_D, r_P))
        print("  max ODE residual  T -> T + c (not a symmetry): %.2f" % r_T)

run_self_tests()

# %%
# --- Settings and figure functions -----------------------------------------------
@dataclass
class Settings:
    c_mode: str = "Least-squares (window)"
    c_custom: float = 0.40
    fit_lo: float = 0.10
    fit_hi: float = 5.0
    t_max: float = 3.0
    tau: float = 0.30
    eps: float = 0.50
    c_trans: float = 0.30
    K_series: int = 4
    t0: float = 1.0
    K_cont: int = 3

MODES = ["Level-matched (c = θ/2)", "Tail-matched (c = θ/√12)",
         "Least-squares (window)", "Custom"]

def resolve_c(S):
    """Return (c, label, is_extra) for the user-selected Varshni variant."""
    if S.c_mode == MODES[0]:
        return C_LEVEL, r"level-matched $c=\theta/2$", False
    if S.c_mode == MODES[1]:
        return C_TAIL, r"tail-matched $c=\theta/\sqrt{12}$", False
    if S.c_mode == MODES[2]:
        c, _ = fit_c(S.fit_lo, S.fit_hi)
        return c, r"least-squares $c=%.3f\,\theta$" % c, True
    return S.c_custom, r"custom $c=%.3f\,\theta$" % S.c_custom, True

def variants(S):
    """Varshni variants to draw: always level and tail matched, plus the user choice if different."""
    out = [dict(label=r"Varshni, level-matched ($c=\theta/2$)", c=C_LEVEL, color=COL["level"], ls="--"),
           dict(label=r"Varshni, tail-matched ($c=\theta/\sqrt{12}$)", c=C_TAIL, color=COL["tail"], ls="-.")]
    c, lbl, extra = resolve_c(S)
    if extra:
        out.append(dict(label="Varshni, " + lbl, c=c, color=COL["user"], ls=":"))
    return out

def _mask_pole(y, t, pole, width=0.03):
    y = np.array(y, dtype=float)
    y[np.abs(t - pole) < width] = np.nan
    return y


def fig_overview(P, S):
    t = np.linspace(1e-3, S.t_max, 900); T = P.theta * t
    fig, ax = plt.subplots(figsize=(8.2, 5.2))
    ax.plot(T, P.E_of_D(D_exact(t)), color=COL["exact"], lw=2.2,
            label=r"exact single-mode BE: $E_\infty-a\coth(\theta/2T)$")
    for v in variants(S):
        ax.plot(T, P.E_of_D(D_varshni(t, v["c"])), color=v["color"], ls=v["ls"], lw=1.7, label=v["label"])
        ax.plot(0, P.E_of_D(D_varshni(0.0, v["c"])), "o", color=v["color"], ms=5)
    ax.plot(T, P.E_inf - P.alpha * T, color=COL["ref"], ls=":", lw=1.2, label=r"classical line $E_\infty-\alpha T$")
    ax.plot(0, P.E0, "o", color=COL["exact"], ms=7)
    ax.axhline(P.E_inf, color="k", lw=0.7, alpha=0.5)
    ax.text(0.99, 0.97, r"$E_\infty$", transform=ax.transAxes, ha="right", va="top")
    ax.set_xlabel(r"$T$ [K]"); ax.set_ylabel(r"$E_g$ [eV]")
    ax.set_title(r"Exact single-mode gap and its Varshni regularizations (dots: $T=0$)")
    ax.legend(loc="lower left", frameon=True)
    fig.tight_layout(); return fig


def fig_parity(P, S):
    tp = np.linspace(0.02, S.t_max, 700); tn = -tp[::-1]; tt = np.concatenate([tn, tp])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))
    ax1.plot(tt, -P.A * D_exact(tt), color=COL["exact"], lw=2.2, label=r"exact ($\mathcal{P}$-odd)")
    for v in variants(S):
        y = _mask_pole(-P.A * D_varshni(tt, v["c"]), tt, -v["c"])
        ax1.plot(tt, y, color=v["color"], ls=v["ls"], lw=1.5, label=v["label"])
        ax1.axvline(-v["c"], color=v["color"], lw=0.6, alpha=0.5)
    ax1.axhline(0, color="k", lw=0.6); ax1.axvline(0, color="k", lw=0.6)
    ax1.set_ylim(-P.A * (S.t_max + 0.6), P.A * (S.t_max + 0.6))
    ax1.set_xlabel(r"$t=T/\theta$ (negative $t$: analytic continuation)")
    ax1.set_ylabel(r"$G=E-E_\infty$ [eV]")
    ax1.set_title(r"Parity $G(-T)=-G(T)$; thin lines: Varshni poles at $T=-\beta$")
    ax1.legend(loc="upper left", fontsize=8)

    floor = 1e-18
    ax2.semilogy(tp, np.abs(-P.A * (D_exact(tp) + D_exact(-tp))) + floor, color=COL["exact"], lw=2.2, label="exact")
    for v in variants(S):
        d = P.A * np.abs(D_varshni(tp, v["c"]) + D_varshni(-tp, v["c"]))
        ax2.semilogy(tp, _mask_pole(d, tp, v["c"], 0.02) + floor, color=v["color"], ls=v["ls"], lw=1.5, label=v["label"])
        ax2.semilogy(tp, 2 * P.A * v["c"]**3 / tp**2, color=v["color"], lw=0.7, alpha=0.5)
    ax2.set_ylim(1e-18, None)
    ax2.set_xlabel(r"$t=T/\theta$"); ax2.set_ylabel(r"parity defect $|G(T)+G(-T)|$ [eV]")
    ax2.set_title(r"Defect: $0$ (to rounding) vs. $2\alpha\theta c^3/T^2$ (thin lines)")
    fig.tight_layout(); return fig


def fig_tail(P, S):
    t = np.linspace(0.1, S.t_max, 900)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.2, 7.2), sharex=True,
                                   gridspec_kw=dict(height_ratios=[3, 1.4]))
    ax1.plot(t, P.A * f_exact(t), color=COL["exact"], lw=2.4, label=r"exact: $f=-a\,L(\theta/2T)$ (Langevin)")
    tm = t > T_SERIES_RADIUS * 1.05
    for K, ls in zip((1, 2, 3), ("-", "--", ":")):
        ax1.plot(t[tm], P.A * (-(D_series(t[tm], K) - t[tm])), color=COL["aux"], ls=ls, lw=1.3,
                 label=r"tail series, $%d$ term%s" % (K, "" if K == 1 else "s"))
    for v in variants(S):
        ax1.plot(t, P.A * f_varshni(t, v["c"]), color=v["color"], ls=v["ls"], lw=1.5, label=v["label"])
    ax1.axhline(0, color="k", lw=0.6)
    ax1.set_ylabel(r"$f=E-E_\infty+\alpha T$ [eV]")
    ax1.set_title(r"Deviation from the classical line: $T^2f''-2f+(2/\alpha)ff'=0$ (exact)")
    ax1.legend(loc="lower right", fontsize=8)
    ax2.semilogy(t, epsilon(t), color="k", lw=1.6)
    ax2.axhline(1.0, color="r", lw=0.8, ls="--")
    ax2.axvline(T_SERIES_RADIUS, color="gray", lw=0.8, ls=":")
    ax2.set_ylabel(r"$\varepsilon=\theta^2/12T^2$"); ax2.set_xlabel(r"$t=T/\theta$")
    ax2.set_title(r"nonlinear/linear ratio; dotted: series radius $t=1/2\pi$", fontsize=10)
    fig.tight_layout(); return fig


def fig_errors(P, S):
    t = np.linspace(0.05, S.t_max, 900)
    fig, ax = plt.subplots(figsize=(8.4, 5.0))
    lines = []
    for v in variants(S):
        dE = 1e3 * P.A * (-(D_varshni(t, v["c"]) - D_exact(t)))            # E_V - E in meV
        ax.plot(t, dE, color=v["color"], ls=v["ls"], lw=1.7, label=v["label"])
        rms, mx, tm = error_stats(v["c"], max(S.fit_lo, 0.05), S.fit_hi)
        lines.append("c=%.3f: rms %.1f meV, max %.1f meV (t=%.2f)" % (v["c"], 1e3 * P.A * rms, 1e3 * P.A * mx, tm))
    ax.axhline(0, color="k", lw=0.7)
    ax.axvspan(0, T_SERIES_RADIUS, color="gray", alpha=0.15)
    ax.text(0.3 * T_SERIES_RADIUS, ax.get_ylim()[1] * 0.9, "series\ndiverges", fontsize=8, ha="center")
    ax.set_xlabel(r"$t=T/\theta$"); ax.set_ylabel(r"$E_V-E_{\rm exact}$ [meV]")
    ax.set_title(r"Error of the asymptote-matched Varshni forms (exact $\alpha$, $E_\infty$)")
    ax.legend(loc="best", fontsize=8)
    ax.text(0.99, 0.02, "\n".join(lines), transform=ax.transAxes, ha="right", va="bottom", fontsize=7.5,
            bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
    fig.tight_layout(); return fig


def fig_orbits(P, S):
    t = np.linspace(0.02, S.t_max, 700); T = P.theta * t
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.9))
    taus = sorted(set([-0.4, -0.2, 0.0, 0.2, 0.4, round(S.tau, 3)]))
    cmap = plt.get_cmap("viridis")
    for i, tau in enumerate(taus):
        y = P.E_of_g(act_dilation(g_exact, tau)(t))
        hl = abs(tau - S.tau) < 1e-9
        ax1.plot(T, y, color=cmap(i / max(len(taus) - 1, 1)), lw=2.6 if hl else 1.3,
                 label=r"$\tau=%.2f$ ($\theta\to%.0f$ K)" % (tau, P.theta * np.exp(tau)))
    ax1.plot(T, P.E_inf - P.alpha * T, color=COL["ref"], ls=":", label=r"classical line (fixed point of $X_D$)")
    ax1.set_title(r"Flow of $X_D=T\partial_T+G\partial_E$: BE curves at fixed $\alpha$")
    ax1.set_xlabel(r"$T$ [K]"); ax1.set_ylabel(r"$E_g$ [eV]"); ax1.legend(fontsize=8, loc="lower left")

    epss = sorted(set([-0.15, 0.0, 0.3, 0.6, 1.0, round(S.eps, 3)]))
    for i, e in enumerate(epss):
        g = act_projective(g_exact, e)(t)
        g = np.where(np.abs(g) > S.t_max + 1.0, np.nan, g)
        hl = abs(e - S.eps) < 1e-9
        lab = r"$\epsilon=%.2f$" % e + (r" (pole at $t=%.1f$)" % (1 / abs(e)) if e < 0 and 1 / abs(e) < S.t_max else "")
        ax2.plot(T, P.E_of_g(g), color=cmap(i / max(len(epss) - 1, 1)), lw=2.6 if hl else 1.3, label=lab)
    ax2.plot(T, P.E_of_g(g_exact(t)), color=COL["exact"], lw=0.8, ls="--")
    ax2.set_ylim(P.E_inf - P.A * (S.t_max + 1.0), P.E_inf + 0.3 * P.A)
    ax2.set_title(r"Flow of $X_P=T^2\partial_T$: shifts $\nu_0$ (fugacity); dashed: Bose ($\epsilon=0$)")
    ax2.set_xlabel(r"$T$ [K]"); ax2.legend(fontsize=8, loc="lower left")
    fig.tight_layout(); return fig


def fig_residuals(P, S):
    t = np.linspace(0.15, max(S.t_max, 1.0), 500)
    tests = [
        (r"exact BE", g_exact, COL["exact"], "-"),
        (r"$X_D$ image ($\tau=%.2f$)" % S.tau, act_dilation(g_exact, S.tau), "tab:cyan", "--"),
        (r"$X_P$ image ($\epsilon=%.2f$)" % S.eps, act_projective(g_exact, S.eps), "tab:olive", "--"),
        (r"translation $T\to T+%.2f\theta$ (not a symmetry)" % S.c_trans, act_translation(g_exact, S.c_trans), "tab:red", "-"),
        (r"translation + $E$-shift (large-$T$ only)", act_translation_shift(g_exact, S.c_trans), COL["aux"], "-."),
        (r"Varshni, level-matched", lambda x: g_varshni(x, C_LEVEL), COL["level"], ":"),
    ]
    fig, ax = plt.subplots(figsize=(8.6, 5.2))
    for lab, g, col, ls in tests:
        r = np.abs(ode_residual(g, t))
        ax.semilogy(t, np.maximum(r, 1e-12), color=col, ls=ls, lw=1.7, label=lab)
    ax.set_ylim(1e-12, 10)
    ax.set_xlabel(r"$t=T/\theta$")
    ax.set_ylabel(r"relative residual of $E''=\Phi E'$")
    ax.set_title("Numerical symmetry test (floor ~1e-9 is finite-difference noise)")
    ax.legend(fontsize=8, loc="center right")
    fig.tight_layout(); return fig


def fig_series(P, S):
    t = np.logspace(np.log10(0.12), np.log10(max(S.t_max, 2.0) * 1.7), 500)
    fig, ax = plt.subplots(figsize=(8.2, 5.0))
    for K in range(0, S.K_series + 1):
        rel = np.abs(D_series(t, K) - D_exact(t)) / D_exact(t)
        ax.loglog(t, np.maximum(rel, 1e-17), lw=1.5, label=r"$K=%d$" % K)
    ax.axvline(T_SERIES_RADIUS, color="k", ls=":", lw=1)
    ax.text(T_SERIES_RADIUS * 1.04, 3e-16, r"$t=1/2\pi$", fontsize=9)
    ax.set_ylim(1e-17, 10)
    ax.set_xlabel(r"$t=T/\theta$"); ax.set_ylabel(r"$|D_K-D|/D$")
    ax.set_title(r"Graded series $D=\sum_k B_{2k}\,t^{1-2k}/(2k)!$: only odd powers of $1/T$")
    ax.legend(ncol=3, fontsize=8)
    fig.tight_layout(); return fig


def fig_continuation(P, S):
    t_end = 0.12
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.4, 7.2), sharex=True,
                                   gridspec_kw=dict(height_ratios=[3, 2]))
    tg = np.linspace(t_end, S.t_max, 800)
    ax1.plot(tg, P.E_of_g(g_exact(tg)), color=COL["exact"], lw=2.4, label="exact")
    cmap = plt.get_cmap("plasma")
    for K in range(0, S.K_cont + 1):
        try:
            tt, gg = continue_from_tail(S.t0, K, t_end)
        except Exception as exc:                                           # pragma: no cover
            print("continuation K=%d failed: %s" % (K, exc)); continue
        col = cmap(K / max(S.K_cont, 1) * 0.85)
        ax1.plot(tt, P.E_of_g(gg), color=col, ls="--", lw=1.5, label=r"tail data $K=%d$" % K)
        ax2.semilogy(tt, np.maximum(1e3 * P.A * np.abs(gg - g_exact(tt)), 1e-12), color=col, lw=1.5)
    for ax in (ax1, ax2): ax.axvline(S.t0, color="gray", ls=":", lw=1)
    ax1.set_ylabel(r"$E_g$ [eV]"); ax1.set_title(r"ODE integrated downward from truncated tail data at $t_0=%.2f$" % S.t0)
    ax1.set_xlim(t_end, S.t_max); ax1.legend(fontsize=8, loc="lower right")
    ax2.set_ylabel(r"$|E_{\rm num}-E_{\rm exact}|$ [meV]"); ax2.set_xlabel(r"$t=T/\theta$")
    fig.tight_layout(); return fig


VIEWS = {
    "1  Overview  E(T)": fig_overview,
    "2  Parity structure": fig_parity,
    "3  Tail, Euler part, ε(T)": fig_tail,
    "4  Varshni errors": fig_errors,
    "5  Symmetry orbits (X_D, X_P)": fig_orbits,
    "6  ODE residual test": fig_residuals,
    "7  Graded (Bernoulli) series": fig_series,
    "8  Continuation from tail": fig_continuation,
}


def summary_html(P, S):
    rows = []
    for v in variants(S):
        c = v["c"]; rms, mx, _ = error_stats(c, max(S.fit_lo, 0.05), S.fit_hi)
        rows.append("<tr><td>%s</td><td>%.4f</td><td>%.1f</td><td>%.4f</td><td>%+.1f</td><td>%.3g</td><td>%.1f / %.1f</td></tr>" % (
            v["label"].replace("$", "").replace("\\", "").replace("{", "").replace("}", ""),
            c, c * P.theta, P.E_inf - P.alpha * c * P.theta,
            1e3 * P.alpha * P.theta * (0.5 - c),
            P.alpha * (c * P.theta) ** 3, 1e3 * P.A * rms, 1e3 * P.A * mx))
    head = ("<b>a</b> = %.4f eV &nbsp; <b>A=αθ</b> = %.4f eV &nbsp; <b>λ=2/α</b> = %.0f K/eV &nbsp; "
            "<b>E(0)</b> = %.4f eV &nbsp; <b>ε(θ)</b> = %.3f, <b>ε(θ/2)</b> = %.3f &nbsp; "
            "<b>ρ</b>: exact = 3, Varshni = 1") % (P.a, P.A, P.lam, P.E0, epsilon(1.0), epsilon(0.5))
    table = ("<table style='border-collapse:collapse;font-size:12px' border='1' cellpadding='3'>"
             "<tr><th>variant</th><th>c/θ</th><th>β [K]</th><th>E_V(0) [eV]</th>"
             "<th>E_V(0)−E(0) [meV]</th><th>1/T² tail coeff. αβ³ [eV K²]</th><th>rms / max err [meV]</th></tr>"
             + "".join(rows) + "</table>")
    return "<div style='font-size:12px'>" + head + "<br><br>" + table + "</div>"

# %%
# --- Interactive dashboard -----------------------------------------------------------
sty = {"description_width": "150px"}
lay = widgets.Layout(width="480px")
mk = lambda cls, **kw: cls(style=sty, layout=lay, continuous_update=False, **kw)

w_view = widgets.Dropdown(options=list(VIEWS), value=list(VIEWS)[0], description="View:", layout=widgets.Layout(width="480px"))
w_E = mk(widgets.FloatSlider, value=3.0, min=0.5, max=6.0, step=0.05, description="E∞ [eV]")
w_al = mk(widgets.FloatSlider, value=4.0, min=0.5, max=12.0, step=0.1, description="α [1e-4 eV/K]")
w_th = mk(widgets.FloatSlider, value=300.0, min=50.0, max=900.0, step=10.0, description="θ [K]")
w_cm = widgets.Dropdown(options=MODES, value=MODES[2], description="Varshni c:", style=sty, layout=lay)
w_cc = mk(widgets.FloatSlider, value=0.40, min=0.05, max=1.2, step=0.005, readout_format=".3f", description="custom c/θ")
w_fw = mk(widgets.FloatRangeSlider, value=(0.10, 5.0), min=0.05, max=8.0, step=0.05, description="fit window t=T/θ")
w_tm = mk(widgets.FloatSlider, value=3.0, min=1.0, max=8.0, step=0.25, description="t_max")
w_tau = mk(widgets.FloatSlider, value=0.30, min=-0.8, max=0.8, step=0.05, description="X_D parameter τ")
w_eps = mk(widgets.FloatSlider, value=0.50, min=-0.3, max=2.0, step=0.05, description="X_P parameter ε")
w_ct = mk(widgets.FloatSlider, value=0.30, min=0.05, max=1.0, step=0.05, description="translation c/θ")
w_Ks = mk(widgets.IntSlider, value=4, min=0, max=8, description="series order K")
w_t0 = mk(widgets.FloatSlider, value=1.0, min=0.4, max=6.0, step=0.1, description="start t0")
w_Kc = mk(widgets.IntSlider, value=3, min=0, max=6, description="tail-data order K")

b_png = widgets.Button(description="Save figure (600 dpi PNG)", button_style="success", icon="download")
b_all = widgets.Button(description="Save all views + CSV (zip)", button_style="info", icon="archive")
out = widgets.Output()

acc = widgets.Accordion(children=[
    widgets.VBox([w_E, w_al, w_th]),
    widgets.VBox([w_cm, w_cc, w_fw, w_tm]),
    widgets.VBox([w_tau, w_eps, w_ct]),
    widgets.VBox([w_Ks, w_t0, w_Kc])])
for i, ttl in enumerate(["Material parameters", "Varshni matching and range",
                         "Symmetry tests (views 5, 6)", "Series / continuation (views 7, 8)"]):
    acc.set_title(i, ttl)
acc.selected_index = 0


def get_state():
    P = Params(E_inf=w_E.value, alpha=w_al.value * 1e-4, theta=w_th.value)
    S = Settings(c_mode=w_cm.value, c_custom=w_cc.value, fit_lo=w_fw.value[0], fit_hi=w_fw.value[1],
                 t_max=w_tm.value, tau=w_tau.value, eps=w_eps.value, c_trans=w_ct.value,
                 K_series=w_Ks.value, t0=w_t0.value, K_cont=w_Kc.value)
    return P, S


def update(_=None):
    P, S = get_state()
    with out:
        clear_output(wait=True)
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                fig = VIEWS[w_view.value](P, S)
            plt.show(fig); plt.close(fig)
            display(HTML(summary_html(P, S)))
        except Exception as exc:                                           # keep the UI alive
            print("Could not draw this view:", repr(exc))


def _deliver(name, data):
    os.makedirs("figures", exist_ok=True)
    path = os.path.join("figures", name)
    with open(path, "wb") as fh: fh.write(data)
    if IN_COLAB:
        colab_files.download(path)
    else:
        print("saved", path)


def _png(fig, dpi=600):
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight"); plt.close(fig)
    return buf.getvalue()


def _tag(P):
    return "Einf%.2f_alpha%.1e_theta%.0f" % (P.E_inf, P.alpha, P.theta)


def csv_bytes(P, S):
    t = np.linspace(1e-3, S.t_max, 600); c_user = resolve_c(S)[0]
    cols = {"t": t, "T_K": P.theta * t, "E_exact_eV": P.E_of_D(D_exact(t)),
            "E_varshni_level_eV": P.E_of_D(D_varshni(t, C_LEVEL)),
            "E_varshni_tail_eV": P.E_of_D(D_varshni(t, C_TAIL)),
            "E_varshni_user_eV": P.E_of_D(D_varshni(t, c_user)),
            "f_exact_eV": P.A * f_exact(t), "epsilon": epsilon(t),
            "dEdT_exact_eV_per_K": np.gradient(P.E_of_D(D_exact(t)), P.theta * t)}
    hdr = ",".join(cols); body = "\n".join(",".join("%.10g" % cols[k][i] for k in cols) for i in range(len(t)))
    return (hdr + "\n" + body + "\n").encode()


def save_current(_):
    P, S = get_state()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fig = VIEWS[w_view.value](P, S)
    _deliver("%s_%s.png" % (w_view.value.split()[0] + "_" + w_view.value.split()[1].lower(), _tag(P)), _png(fig))


def save_all(_):
    P, S = get_state(); buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf, warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for name, fn in VIEWS.items():
            zf.writestr("view_%s.png" % name.split()[0], _png(fn(P, S)))
        zf.writestr("curves_%s.csv" % _tag(P), csv_bytes(P, S))
    _deliver("bandgap_symmetry_%s.zip" % _tag(P), buf.getvalue())


for w in (w_view, w_E, w_al, w_th, w_cm, w_cc, w_fw, w_tm, w_tau, w_eps, w_ct, w_Ks, w_t0, w_Kc):
    w.observe(update, names="value")
b_png.on_click(save_current); b_all.on_click(save_all)

display(widgets.VBox([w_view, acc, widgets.HBox([b_png, b_all])]), out)
update()

All self-tests passed.
  max ODE residual  exact: 7.6e-10   X_D image: 1.7e-08   X_P image: 2.8e-09
  max ODE residual  T -> T + c (not a symmetry): 0.57


Output()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>